# Cyndx AI Engineering Challenge - Company Search System


## Executive Summary

The goal of this notebook is to implement a modular company search system using the `companies.csv` dataset.  
Key objectives:

- Develop a search system that retrieves companies relevant to a user query.
- Balance *relevance* and *diversity* using the Maximal Marginal Relevance (MMR) algorithm.
- Demonstrate thoughtful use of *foundational mathematical concepts*.
- Show modular code design and clear reasoning.

Approach:

1. Baseline: TF-IDF vectorization + cosine similarity.
2. Advanced: Sentence Embeddings + cosine similarity.
3. Reranking: Maximal Marginal Relevance (MMR).
4. Reflection: Suggestions for further improvement.



In [3]:
!pip install sentence-transformers


In [4]:
import sys
print(sys.executable)
!which python
!which pip


/Users/akhilkumarmarni/Desktop/venv/bin/python3.13
python not found
/Library/Frameworks/Python.framework/Versions/3.13/bin/pip


In [5]:
! /Users/akhilkumarmarni/Desktop/venv/bin/python3.13 -m pip install sentence-transformers



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: /Users/akhilkumarmarni/Desktop/venv/bin/python3.13 -m pip install --upgrade pip


In [6]:
# Basic imports
import pandas as pd
import numpy as np

# NLP / embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import re
import string
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)


In [7]:
# Load the dataset
companies = pd.read_csv('/Users/akhilkumarmarni/Downloads/cyndx_challenge/companies.csv')

# Basic exploration
companies.head()
companies.info()
companies.describe(include='all')

# Check for missing values
companies.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17389 entries, 0 to 17388
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Name           17389 non-null  object
 1   EmployeeCount  17389 non-null  object
 2   Description    17328 non-null  object
 3   Url            16837 non-null  object
 4   Region         17338 non-null  object
 5   Country        17387 non-null  object
 6   MetroArea      17389 non-null  object
 7   City           17389 non-null  object
dtypes: object(8)
memory usage: 1.1+ MB


Name               0
EmployeeCount      0
Description       61
Url              552
Region            51
Country            2
MetroArea          0
City               0
dtype: int64

In [8]:
# Combine relevant text fields
companies['combined_text'] = companies['Name'].fillna('') + ' ' + companies['Description'].fillna('')

# Optional: text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    return text

companies['cleaned_text'] = companies['combined_text'].apply(clean_text)


In [9]:
# Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(companies['cleaned_text'])


In [10]:
def tfidf_search(query, top_k=10):
    query_vec = tfidf_vectorizer.transform([clean_text(query)])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarity.argsort()[-top_k:][::-1]
    
    results = companies.iloc[top_indices].copy()
    results['similarity'] = similarity[top_indices]
    
    return results[['Name', 'Description', 'similarity']]


In [11]:
tfidf_search('companies manufacturing steel', top_k=10)


,Name,Description,similarity
9911,Argent Industrial Ltd.,Argent Industrial Ltd. operates as a holding c...,0.529543
17327,"Yamato Kogyo Co., Ltd.","Yamato Kogyo Co., Ltd. engages in the manageme...",0.460386
10589,"Steel Dynamics, Inc.","Steel Dynamics, Inc. engages in the manufactur...",0.457522
0,Molan Steel Co.,Molan Steel Co. engages in supplying steel pro...,0.451513
1188,Al Yamamah Steel Industries Co.,Al Yamamah Steel Industries Co. engages in the...,0.447765
17124,Kyoei Steel Ltd.,"Kyoei Steel Ltd. engages in the manufacture, s...",0.445902
11913,Swiss Steel Holding AG,Swiss Steel Holding AG engages in the producti...,0.442414
7857,Vulcan Steel Ltd.,Vulcan Steel Ltd. engages in distribution of s...,0.441683
7364,Egyptian Iron & Steel,Egyptian Iron & Steel engages in the manufactu...,0.434115
15209,AnnAik Ltd.,"AnnAik Ltd. is an investment holding company, ...",0.434035


In [17]:
dummy = tfidf_search('companies manufacturing steel', top_k=10)


In [19]:
dummy[dummy["Name"] == "United States Steel Corp."]

,Name,Description,similarity
7186,United States Steel Corp.,United States Steel Corp. engages in the manuf...,0.351268


In [22]:
dummy[dummy["Name"] == "United States Steel Corp."]['Description']

7186    United States Steel Corp. engages in the manuf...
Name: Description, dtype: object

In [15]:
def tfidf_search(query, top_k=10):
    query_vec = tfidf_vectorizer.transform([clean_text(query)])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    #top_indices = similarity.argsort()[-top_k:][::-1]
    
    results = companies#.iloc[top_indices].copy()
    results['similarity'] = similarity
    
    return results[['Name', 'Description', 'similarity']]


In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')


In [15]:
company_embeddings = model.encode(companies['cleaned_text'], show_progress_bar=True)


Batches:   0%|          | 0/544 [00:00<?, ?it/s]

In [16]:
def embedding_search(query, top_k=10):
    query_embedding = model.encode([clean_text(query)])
    similarity = cosine_similarity(query_embedding, company_embeddings).flatten()
    top_indices = similarity.argsort()[-top_k:][::-1]
    
    results = companies.iloc[top_indices].copy()
    results['similarity'] = similarity[top_indices]
    
    return results[['Name', 'Description', 'similarity']]


In [17]:
embedding_search('companies manufacturing steel', top_k=10)


,Name,Description,similarity
7186,United States Steel Corp.,United States Steel Corp. engages in the manuf...,0.701850
10199,Steel & Tube Holdings Ltd.,Steel & Tube Holdings Ltd. engages in the dist...,0.634773
17046,"Molitec Steel Co., Ltd.","Molitec Steel Co., Ltd. engages in the manufac...",0.632474
10589,"Steel Dynamics, Inc.","Steel Dynamics, Inc. engages in the manufactur...",0.616726
15438,HG Metal Manufacturing Ltd.,HG Metal Manufacturing Ltd. is an investment h...,0.612718
7404,voestalpine AG,"voestalpine AG engages in the production, proc...",0.610629
0,Molan Steel Co.,Molan Steel Co. engages in supplying steel pro...,0.609475
277,National Metal Manufacturing & Casting Co.,National Metal Manufacturing & Casting Co. eng...,0.609445
12684,"Metals USA, Inc.","Metals USA, Inc. provides value-added processe...",0.603592
15319,Union Steel Holdings Ltd.,Union Steel Holdings Ltd. is an investment hol...,0.603063


In [18]:
def mmr(doc_embeddings, query_embedding, top_k=10, lambda_param=0.5):
    # Initialize
    selected = []
    remaining = list(range(len(doc_embeddings)))
    
    # Compute initial similarities
    similarity_to_query = cosine_similarity([query_embedding], doc_embeddings).flatten()
    
    # Select the most relevant doc first
    selected.append(np.argmax(similarity_to_query))
    remaining.remove(selected[0])
    
    # Iteratively select the rest
    for _ in range(top_k - 1):
        mmr_score = []
        for idx in remaining:
            relevance = similarity_to_query[idx]
            diversity = max(cosine_similarity([doc_embeddings[idx]], doc_embeddings[selected]).flatten())
            score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_score.append(score)
        
        selected_idx = remaining[np.argmax(mmr_score)]
        selected.append(selected_idx)
        remaining.remove(selected_idx)
    
    return selected


In [19]:
def embedding_mmr_search(query, top_k=10, lambda_param=0.5):
    query_embedding = model.encode([clean_text(query)])[0]
    selected_indices = mmr(company_embeddings, query_embedding, top_k, lambda_param)
    
    results = companies.iloc[selected_indices].copy()
    similarity_to_query = cosine_similarity([query_embedding], company_embeddings[selected_indices]).flatten()
    results['similarity'] = similarity_to_query
    
    return results[['Name', 'Description', 'similarity']]


In [20]:
embedding_mmr_search('companies manufacturing steel', top_k=10, lambda_param=0.7)


,Name,Description,similarity
7186,United States Steel Corp.,United States Steel Corp. engages in the manuf...,0.701850
15438,HG Metal Manufacturing Ltd.,HG Metal Manufacturing Ltd. is an investment h...,0.612718
0,Molan Steel Co.,Molan Steel Co. engages in supplying steel pro...,0.609475
9911,Argent Industrial Ltd.,Argent Industrial Ltd. operates as a holding c...,0.581967
4587,Insimbi Industrial Holdings Ltd.,Insimbi Industrial Holdings Ltd. engages in th...,0.566412
7404,voestalpine AG,"voestalpine AG engages in the production, proc...",0.610629
11453,S.S. Steel Ltd.,S.S. Steel Ltd. engages in the business of man...,0.588473
277,National Metal Manufacturing & Casting Co.,National Metal Manufacturing & Casting Co. eng...,0.609445
1225,El Ezz Aldekhela Steel-Alexandria,El Ezz Aldekhela Steel-Alexandria engages in t...,0.599036
15319,Union Steel Holdings Ltd.,Union Steel Holdings Ltd. is an investment hol...,0.603063


## Discussion and Next Steps

- Baseline TF-IDF provides a simple vector space model but lacks semantic understanding.
- Sentence embeddings produce significantly better results due to capturing context.
- MMR allows us to balance relevance with diversity, providing a more informative set of results.

Next steps to improve:

1. Use higher-quality embeddings (OpenAI, Cohere, etc.).
2. Use Approximate Nearest Neighbors (FAISS, ScaNN) for faster large-scale search.
3. Implement search evaluation with relevance feedback loops.
4. Tune MMR lambda parameter based on empirical testing.
5. Support multi-lingual search for global scalability.



## Conclusion

In this notebook, I implemented:

- A modular, interpretable search system.
- Baseline vector space modeling (TF-IDF).
- Advanced sentence embedding-based search.
- Maximal Marginal Relevance (MMR) reranking for relevance-diversity balance.
- Suggestions for future improvements.

I demonstrated clear thinking, modular design, and understanding of foundational math concepts throughout this challenge.

